# JSON to SBML conversion

Converts the `ecBSU1` and `iBsu1147R` models from COBRApy JSON format to SBML (`.xml`).

## Define source/output paths

In [ ]:
import os
from cobra.io import load_json_model, write_sbml_model, read_sbml_model

models_dir = os.path.join(os.getcwd(), "..", "data", "models", "00_initial")
sbml_dir = os.path.join(os.getcwd(), "..", "data", "models", "01_SBML_models")
os.makedirs(sbml_dir, exist_ok=True)

conversions = {
    "ecBSU1": {
        "json_path": os.path.join(models_dir, "ecBSU1", "iBsu1147_irr_enz_constraint_adj.json"),
        "sbml_path": os.path.join(sbml_dir, "ecBSU1.xml"),
    },
    "iBsu1147R": {
        "json_path": os.path.join(models_dir, "iBsu1147Revised", "iBsu1147_modify.json"),
        "sbml_path": os.path.join(sbml_dir, "iBsu1147R.xml"),
    },
}

## Convert to SBML

In [ ]:
for name, paths in conversions.items():
    print(f"[INFO] Loading {name} from {paths['json_path']}")
    model = load_json_model(paths["json_path"])
    print(f"[INFO] {name}: {len(model.genes)} genes, {len(model.metabolites)} metabolites, {len(model.reactions)} reactions")

    # A few metabolites have an empty compartment field, which write_sbml_model turns into
    # an invalid <Compartment> that breaks downstream SBML readers. Infer it from the id
    # suffix instead (e.g. "cppp9_c" -> "c").
    fixed = []
    for met in model.metabolites:
        if not met.compartment:
            inferred_compartment = met.id.rsplit("_", 1)[-1]
            met.compartment = inferred_compartment
            model.compartments.setdefault(inferred_compartment, inferred_compartment)
            fixed.append(met.id)
    if fixed:
        print(f"[INFO] Fixed missing compartment for {len(fixed)} metabolite(s) in {name}: {fixed}")

    write_sbml_model(model, paths["sbml_path"])
    print(f"[FINISHED] {name} written to {paths['sbml_path']}\n")